<a href="https://colab.research.google.com/github/Shashith240/Statistical-Learning-e22240/blob/main/E22240_Assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>



## 1. Deriving the Marginal Density

To find the marginal density $p(x_i)$, we apply the **Law of Total Probability** by summing out the latent cluster variable $C_i$ across all possible $K$ groups:

$$p(x_i) = \sum_{k=1}^{K} p(x_i, C_i = k)$$

Using the definition of conditional probability, we can factor the joint density as $p(x_i, C_i = k) = P(C_i = k) \, p(x_i \mid C_i = k)$. Substituting the given model specifications:

* $P(C_i = k) = \phi_k$ (the cluster prior probability)
* $p(x_i \mid C_i = k) = \mathcal{N}(x_i \mid \mu_k, \Sigma_k)$ (the conditional Gaussian density)

We arrive directly at the marginal density formula:


$$p(x_i) = \sum_{k=1}^{K} \phi_k \mathcal{N}(x_i \mid \mu_k, \Sigma_k)$$

### Interpretation

This density is called a **Gaussian mixture density** because it expresses the overall probability distribution of the data as a convex combination (a weighted blend) of $K$ distinct multivariate Gaussian component distributions, where the weights $\phi_k$ are non-negative and sum to 1.

---

## 2. Deriving the Posterior Cluster Probability (Responsibility)

For a fixed observation $x_i$, we apply **Bayes' Rule** to find the probability that the data point belongs to cluster $k$:

$$P(C_i = k \mid X_i = x_i) = \frac{p(X_i = x_i \mid C_i = k) P(C_i = k)}{p(x_i)}$$

By substituting the law of total probability formulation derived in Task 1 for the denominator $p(x_i)$, we get:

$$P(C_i = k \mid X_i = x_i) = \frac{p(X_i = x_i \mid C_i = k) P(C_i = k)}{\sum_{j=1}^{K} p(X_i = x_i \mid C_i = j) P(C_i = j)}$$

Now, substituting the specific Gaussian distributions and prior parameters:

$$\gamma_{ik} = P(C_i = k \mid X_i = x_i) = \frac{\phi_k \mathcal{N}(x_i \mid \mu_k, \Sigma_k)}{\sum_{j=1}^{K} \phi_j \mathcal{N}(x_i \mid \mu_j, \Sigma_j)}$$

### Interpretation

The quantity $\gamma_{ik}$ is interpreted as a **posterior probability** because it revises our initial "belief" (the prior probability $\phi_k$) about which cluster the point belongs to *after* observing the actual data values $x_i$.

---

## 3. One-Hot Encoding of the Latent Cluster Variable

Let $Z_i = [Z_{i1}, Z_{i2}, \dots, Z_{iK}]^T$ be a one-hot encoded vector where $Z_{ik} = 1$ if $C_i = k$ and $0$ otherwise.

### Step A: Expected Value of a Single Component

By definition, the conditional expectation of a discrete random variable is the sum of its possible values weighted by their conditional probabilities:

$$E[Z_{ik} \mid X_i = x_i] = (1) \cdot P(Z_{ik} = 1 \mid X_i = x_i) + (0) \cdot P(Z_{ik} = 0 \mid X_i = x_i)$$

Since the event $\{Z_{ik} = 1\}$ is identical to the event $\{C_i = k\}$, this simplifies directly to:

$$E[Z_{ik} \mid X_i = x_i] = P(C_i = k \mid X_i = x_i) = \gamma_{ik}$$

### Step B: Expected Value of the Full Vector

Applying the expectation operator element-wise across the column vector $Z_i$:

$$E[Z_i \mid X_i = x_i] = \begin{bmatrix} E[Z_{i1} \mid X_i = x_i] \\ E[Z_{i2} \mid X_i = x_i] \\ \vdots \\ E[Z_{iK} \mid X_i = x_i] \end{bmatrix} = \begin{bmatrix} \gamma_{i1} \\ \gamma_{i2} \\ \vdots \\ \gamma_{iK} \end{bmatrix}$$

### Conclusion

Therefore, the **soft cluster assignment** vector in a Gaussian mixture model is mathematically equivalent to the conditional expectation $E[Z_i \mid X_i = x_i]$.

---

## 4. From Soft Assignment to Hard Clustering

The key differences between soft and hard clustering within this conditional framework are:

* **Soft Clustering ($E[Z_i \mid X_i = x_i]$):** Provides a probabilistic, fractional assignment vector. It acknowledges ambiguity by mapping data point $x_i$ to *all* clusters simultaneously, capturing the exact confidence levels (e.g., a point sitting right between two clusters might be assigned as $50\%$ Cluster 1 and $50\%$ Cluster 2).
* **Hard Clustering ($\hat{C}_i = \text{argmax}_{1 \le k \le K} \gamma_{ik}$):** Provides a deterministic, discrete classification. It forces a decision by mapping the data point strictly to a single cluster—specifically, the one with the maximum a posteriori (MAP) probability—discarding any information about relative cluster ambiguity.

---

## 5. Conditional Expectation of the Observation Given the Cluster

### Proof that $E[X_i \mid C_i = k] = \mu_k$

Given the model definition, the conditional distribution of $X_i$ given that it belongs to cluster $k$ is standard multivariate normal: $X_i \mid C_i = k \sim \mathcal{N}(\mu_k, \Sigma_k)$.

The standard property for the mean of a multivariate normal distribution dictates that:

$$E[X_i \mid C_i = k] = \int_{\mathbb{R}^d} x_i \, \mathcal{N}(x_i \mid \mu_k, \Sigma_k) \, dx_i = \mu_k$$

### Interpretation

Because $E[X_i \mid C_i = k]$ evaluates directly to $\mu_k$, the parameter vector $\mu_k$ acts as the balancing center of mass for all points generated by component $k$. Thus, it is interpreted as the **geometric center (or centroid)** of cluster $k$.